### Type organization seats (oxjob #2437)

Writes `openalex.works.work_author_types`'s `organization` rows from two kinds of evidence, each of which passed its own bar (organization bar v3, oxjobs #2437 `work/org/README.md`): a DataCite creator whose ROR id names the creator itself, in DiSSCo's specimen records (the museum or herbarium that holds the specimen), and a group name (committee, consortium, collaboration and the like) a model has confirmed. Runs in its own weekly job, outside End 2 End: it reads all of `openalex.datacite.datacite_items`. The night after a run, End 2 End's Authorships lays the types on and the works whose seats changed re-stamp.

1. `openalex.authors.type_org_byline_name`: the junk and person guards of oxjobs #2437 `work/build/org_names.py`, a Python UDF **generated** by `work/build/patch_walden_typer.py` (`--check` fails when they differ).
2. ROR creators: in the latest DataCite record of each DOI, the creators with nameType Organizational, a nameIdentifier of scheme ROR and no ORCID (a person with their employer's ROR id filed on them is not an organization), with the ROR ids other creators of the record give as their affiliation.
3. ROR seats: the work with that DOI in tonight's `openalex_works_base`, and the byline seat whose name equals the creator's name (lowercased, punctuation and spacing folded), kept only when that name is one of the ROR record's own names (the ROR id names the author, not an affiliation), no other creator of the record names the organization as their affiliation (an address line filed as a creator), and the record comes from DataCite client DiSSCo (`ylqb.ybhfwy`): other clients' ROR creators failed the bar (institutional repositories list the university as the creator of theses and articles).
4. One guard verdict per distinct ROR seat name (the only statement that calls the UDF).
5. Name seats: seats of tonight's base whose printed name has the answer `organization` in `openalex.authors.org_name_confirmations`: one answer per distinct name the guards proposed as a group, given by google/gemini-4-hydrogen with the bar's rubric (oxjobs #2437 `work/org/confirm_org_names.py`). A name with no answer there stays untyped; new names are confirmed outside this job.
6. Typed seats: ROR seats (rule `ror`) first, then name seats (rule `group_name_confirmed`), except seats the agent typing has typed `software-agent` (agents stay agents).
7. Write: `INSERT INTO ... REPLACE WHERE type = 'organization'` (software-agent rows belong to the nightly confirmer) when the change from the last run is within its lines (lost max(10,000, 10%), added max(100,000, 10%)); otherwise the run fails and the rows stay. A planned change (the first fill, ~13.7M seats) runs with the job parameter `allow_large_change` = `true`.

Revert: take the task out of the weekly job; `DELETE FROM openalex.works.work_author_types WHERE type = 'organization'` clears the rows (the next End 2 End re-stamps those works, pre-cleared). The before-state of the first fill: `openalex.transient.oxjob2437_org_manifest_night1`.

In [ ]:
-- oxjob #2437: GENERATED from oxjobs #2437 work/build/org_names.py (org-names-v2) by work/build/patch_walden_typer.py.
-- Edit the module and its tests, then regenerate; don't edit this cell by hand. Only its junk and person guards
-- decide here: a seat is typed only by rule 'ror'.
CREATE OR REPLACE FUNCTION openalex.authors.type_org_byline_name(s STRING, declared BOOLEAN, ror BOOLEAN)
RETURNS STRUCT<type: STRING, rule: STRING>
LANGUAGE PYTHON
DETERMINISTIC
COMMENT 'oxjob #2437: types a printed byline name organization when its source gives a ROR id (or declares or names a group), after junk and person guards; else null. Source + tests: oxjobs #2437 work/build/org_names.py'
AS $$
"""Organization typing of byline seats (oxjob #2437, 21 Nov 2026). Tests: test_org_names.py.

A seat is typed `organization` when its printed name is a group: a collaboration, consortium, committee, society,
company, agency, institute, university, team or the like. Evidence, strongest first:
1. `ror`: the source gives the creator a ROR id (DataCite nameIdentifier, scheme ROR).
2. `declared`: the source says the creator is an organization (DataCite nameType Organizational) and the name passes
   the junk and person checks below.
3. `group_name`: the printed name holds a group word ("Collaboration", "Study Group", "Committee", "Ltd") and no person.
Crossref's name-only author field is NOT evidence by itself: publishers put people with one name ("Priyanka"),
journals and junk ("&NA;", "Anonymous") there. A Crossref name-only seat is typed only by rule 3.

Never typed: placeholders (Anonymous, Unknown, N/A, none), a person on behalf of a group ("J. Smith for the X
Consortium"), a person with an affiliation pasted into the name ("John Smith, University of X"), sentence fragments.
Precision first: anything uncertain stays null (= not stated).
"""

import re
import unicodedata

RULE_VERSION = "org-names-v2"

# Group words, by language; each must stand as a whole word.
GROUP_WORDS = [
    # research collaborations and groups
    r"collaboration", r"consortium", r"consortia", r"konsortium", r"consorcio", r"consórcio",
    r"study\s+group", r"working\s+group", r"writing\s+group", r"steering\s+group", r"research\s+group",
    r"interest\s+group", r"expert\s+group", r"task\s+force", r"working\s+party", r"investigators",
    r"trialists", r"study\s+team", r"research\s+team", r"project\s+team", r"collaborative\s+group",
    r"collaborators", r"network", r"netzwerk", r"réseau", r"red\s+de", r"panel", r"registry", r"group",
    r"gruppe", r"groupe", r"grupo", r"gruppo", r"team",
    # bodies
    r"committee", r"comité", r"comite", r"comitato", r"komitee", r"ausschuss", r"commission", r"commissie",
    r"council", r"conseil", r"consejo", r"board", r"secretariat", r"secrétariat", r"office", r"bureau",
    r"authority", r"agency", r"agence", r"agencia", r"ministry", r"ministère", r"ministerio", r"ministério",
    r"ministerium", r"department", r"departamento", r"département", r"government", r"gouvernement",
    r"parliament", r"senate", r"congress", r"assembly", r"court", r"tribunal",
    # associations
    r"society", r"société", r"sociedad", r"sociedade", r"società", r"gesellschaft", r"association",
    r"associação", r"asociación", r"associazione", r"verband", r"verein", r"federation", r"fédération",
    r"federación", r"union", r"alliance", r"league", r"academy", r"académie", r"academia", r"akademie",
    r"college\s+of", r"royal\s+college", r"institute", r"institut", r"instituto", r"istituto",
    r"university", r"universit[éàa]", r"universidad", r"universidade", r"universität", r"hochschule",
    r"hospital", r"hôpital", r"klinikum", r"clinic", r"laboratory", r"laboratoire", r"laboratorio",
    r"foundation", r"fondation", r"fundación", r"fundação", r"fondazione", r"stiftung", r"trust",
    r"(?:research\s+)?cent(?:re|er)(?:\s+(?:for|of))?", r"centro", r"zentrum", r"museum", r"library", r"bibliothèque", r"archive[s]?",
    r"observatory", r"initiative", r"programme", r"program", r"project", r"projekt", r"projet",
    r"organi[sz]ation", r"organización", r"organização", r"organisatie",
    # editorial bodies
    r"editorial\s+(?:board|office|team|staff|committee)", r"editors", r"redazione", r"rédaction", r"redaktion",
    r"redaktsiia", r"redacción", r"redação",
    # companies
    r"corporation", r"company", r"co\.,?\s+ltd", r"incorporated",
]
# Company suffixes count only at the end of a name of two or more words ("Philips N.V."); at the start they are
# a person's initials ("N.V. Popova", "S.A. Grishin": org bar v1).
COMPANY_END = re.compile(r"\S+\s+\S*.*?(?:,\s*)?(?<![\w’'])(?:inc\.?|ltd\.?|llc|plc|gmbh|ag|s\.a\.?|s\.p\.a\.?|corp\.?|limited|"
                         r"b\.v\.?|n\.v\.?|pty(?:\s+ltd)?|s\.r\.l\.?|s\.a\.s\.?|oy|ab|as|a/s|kg|sa|srl|spa|bv|nv)\s*$")
GROUP_RE = re.compile(r"(?<![\w’'])(?:" + "|".join(GROUP_WORDS) + r")(?![\w’'])")
# A group word that is also a surname or a word in personal names needs a second group word (or ROR / declared).
WEAK_GROUP = re.compile(r"(?<![\w’'])(?:union|trust|league|court|board|office|bureau|panel|team|network|clinic|"
                        r"ag|editors|program|project|projekt|projet|assembly|congress|senate|group|comite|comité|federation|"
                        r"gruppe|groupe|grupo|gruppo|limited|archives?|library|centre|center|centro)(?![\w’'])")

PLACEHOLDER = re.compile(r"^(?:anonymous|anon\.?|anonym|anónimo|anonyme|unknown|n/?a|none|null|not\s+available|"
                         r"&na;|-+|\.+|author|authors|various|various\s+authors|et\s+al\.?|editor|editors|ㅤ|"
                         r"no\s+author|sin\s+autor|unbekannt|inconnu|redazione|rédaction|redaktion|"
                         r"gbif\.org\s+user|:unav|:unkn|:none|:null|:tba|:etal|nn|xx|xxx|n\.n\.|test|admin|user)$")  # GBIF downloads credit an unnamed user
# A person beside the group: "on behalf of", "for the", "and the", or a person's name before a comma.
PERSON_BESIDE = re.compile(r"(?<![\w’'])(?:on\s+behalf\s+of|in\s+the\s+name\s+of|for\s+the|and\s+the|"
                           r"members\s+of|represented\s+by|et\s+al\.?|edited\s+by|by\s+the|"
                           r"and\s+collaborators|and\s+colleagues)(?![\w’'])")
ORCID = re.compile(r"\d{4}-\d{4}-\d{4}-\d{3}[\dxX]")
EMAIL = re.compile(r"@[\w-]+\.")
PERSON_HEAD = re.compile(r"^\s*(?:dr\.?\s+|prof\.?\s+)?[A-ZÀ-Ý][a-zà-ÿ'’-]+(?:\s+[A-Z]\.?)*(?:\s+[A-ZÀ-Ý][a-zà-ÿ'’-]+){1,2}\s*[,;(]")
FAMILY_GIVEN_PAREN = re.compile(r"^\s*[A-ZÀ-Ý][\w'’-]+(?:\s+[A-ZÀ-Ý][\w'’-]+)?,\s*[A-ZÀ-Ý][\w.'’ -]*\(")
INITIALS_NAME = re.compile(r"^\s*(?:[A-Z]\.\s?){1,3}\s*[A-ZÀ-Ý][a-zà-ÿ'’-]+")  # "N.V. Popova", "S.A. Arden"
PLACE_ONLY = re.compile(r"(?<![\w’'])(?:russian\s+federation|united\s+states|people'?s\s+republic)(?![\w’'])")


def norm(s: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s or "")).strip()


def type_org_name(raw: str, declared: bool = False, ror: bool = False) -> tuple[str | None, str]:
    """(type, rule) for one printed name. declared = the source says Organizational; ror = the source gives a ROR id."""
    s = norm(raw)
    low = s.lower().strip(" .,;:")
    if not low or len(low) < 2:
        return None, "empty"
    if PLACEHOLDER.match(low):
        return None, "placeholder"
    if ORCID.search(s) or EMAIL.search(s):
        return None, "person_signs"
    m = PERSON_BESIDE.search(low)
    if m and not GROUP_RE.search(low[:m.start()]):  # "J. Smith for the X Consortium"; "Writing Committee for the X" is a group
        return None, "person_beside_group"
    if len(s.split()) > 20:
        return None, "fragment"
    groups = GROUP_RE.findall(low)
    if COMPANY_END.match(low):
        groups.append("company")
    strong_groups = [g for g in groups if not WEAK_GROUP.fullmatch(g)]
    head = re.split(r"[,;(]", s, maxsplit=1)[0].lower()
    # "John Smith, University of X", "Ana Laura Zurek; Universidad ...", "Shimizu, Thomas (University of Maryland)"
    if groups and (PERSON_HEAD.match(s) and not GROUP_RE.search(head) or FAMILY_GIVEN_PAREN.match(s)):
        return None, "person_then_affiliation"
    if INITIALS_NAME.match(s) and not GROUP_RE.search(head):
        return None, "initials_name"
    if PLACE_ONLY.search(low) and len(strong_groups) <= 1 and not ror:
        return None, "place"
    if ror:
        return "organization", "ror"
    if declared:
        acronym = re.fullmatch(r"[A-Z0-9&.\-]{2,12}", s) and sum(ch.isalpha() for ch in s) >= 2
        camel = re.fullmatch(r"[A-Za-z0-9]+", s) and re.search(r"[a-z][A-Z]", s)
        if groups or acronym or camel:
            return "organization", "declared"
        return None, "declared_without_group_word"  # "Master, Daniel M.", "Pedro U. Lima" (org bar v1)
    if strong_groups or len(groups) >= 2:
        return "organization", "group_name"
    if groups:
        return None, "weak_group_word_only"
    return None, "no_group_word"


# SQL prefilter (Java regex on LOWER(raw_author_name)): the group words without boundaries.
SQL_PREFILTER = "|".join(
    ["collaborat", "consorti", "konsortium", "consórcio", "group", "gruppe", "grupo", "gruppo", "groupe", "task force",
     "working party", "investigators", "trialists", "team", "network", "netzwerk", "réseau", "panel", "registry",
     "committee", "comit", "komitee", "ausschuss", "commission", "commissie", "council", "conseil", "consejo", "board",
     "secretar", "office", "bureau", "authority", "agenc", "minist", "department", "departamento", "département",
     "government", "gouvernement", "parliament", "senate", "congress", "assembly", "court", "tribunal", "societ",
     "sociedad", "gesellschaft", "associa", "asociación", "verband", "verein", "federa", "fédération", "union",
     "alliance", "league", "academ", "akademie", "college", "institu", "istituto", "universi", "hochschule",
     "hospital", "hôpital", "klinikum", "clinic", "laborat", "foundation", "fondation", "fundaç", "fundación",
     "fondazione", "stiftung", "trust", "centre", "center", "centro", "zentrum", "museum", "library", "biblioth",
     "archive", "observator", "initiative", "program", "project", "projekt", "projet", "organi", "editor", "redaz",
     "rédaction", "redaktion", "redaktsiia", "redacción", "redação", "inc", "ltd", "llc", "plc", "gmbh", "\\bag\\b",
     "s\\.a\\.", "s\\.p\\.a\\.", "corp", "company", "co\\.", "limited", "b\\.v\\.", "n\\.v\\.", "pty", "s\\.r\\.l\\."])

t, r = type_org_name(s, bool(declared), bool(ror))
return {"type": t, "rule": r}
$$

In [ ]:
-- oxjob #2437: the typed-seat table (same definition as CreateWorkAuthorships' cell), and the name confirmer's answers
-- (filled outside this job; created empty here so the job never fails before the first fill).
CREATE TABLE IF NOT EXISTS openalex.works.work_author_types (
  work_id         BIGINT,
  author_sequence BIGINT,
  raw_author_name STRING,
  type            STRING,
  rule            STRING,
  rule_version    STRING,
  evidence        STRING
)
COMMENT 'oxjob #2437: author type per byline seat (work_id, author_sequence as in work_authors); one row per typed seat, no row = not stated. Read by CreateWorkAuthorships.'
CLUSTER BY (work_id);

CREATE TABLE IF NOT EXISTS openalex.authors.org_name_confirmations (
  raw_author_name STRING,
  label           STRING,
  reason          STRING,
  model           STRING,
  rubric          STRING,
  n_seats         BIGINT,
  confirmed_at    TIMESTAMP
)
COMMENT 'oxjob #2437: one model answer per distinct byline name the organization rules propose as a group (label organization | human | mixed | affiliation_or_host | not_a_name | software_agent | unsure). Read by TypeOrganizationSeats: only organization types a seat.'

In [ ]:
-- oxjob #2437: organization creators with a ROR id in the latest DataCite record of each DOI (a DOI any of whose records gives
-- a creator a ROR id is a candidate; the latest record decides). A ROR id counts only on a creator declared Organizational
-- with no ORCID: on a person it is their employer's. Also the ROR ids and names the record's other creators give as their
-- affiliation. ~4 min on a 4X-Large: the whole of datacite_items.
CREATE OR REPLACE TABLE openalex.authors.org_byline_ror_creators
COMMENT 'oxjob #2437: DataCite creators declared Organizational with a ROR id and no ORCID, latest record per DOI. Rebuilt by TypeOrganizationSeats each run.'
AS
WITH cand AS (
  SELECT DISTINCT LOWER(id) AS doi FROM openalex.datacite.datacite_items
  WHERE EXISTS(attributes.creators, c -> EXISTS(c.nameIdentifiers, i -> LOWER(i.nameIdentifierScheme) = 'ror'))
),
latest AS (
  SELECT LOWER(d.id) AS doi, d.attributes.creators AS creators, d.relationships.client.data.id AS client_id,
         ROW_NUMBER() OVER (PARTITION BY LOWER(d.id) ORDER BY d.attributes.updated DESC NULLS LAST,
                                                              d.ingested_at DESC NULLS LAST) AS rn
  FROM openalex.datacite.datacite_items d JOIN cand ON cand.doi = LOWER(d.id)
)
SELECT doi, client_id, pos AS creator_position, c.name AS creator_name,
       ARRAY_DISTINCT(FILTER(TRANSFORM(c.nameIdentifiers,
         i -> CASE WHEN LOWER(i.nameIdentifierScheme) = 'ror'
                   THEN REGEXP_EXTRACT(LOWER(i.nameIdentifier), '(0[a-z0-9]{6}[0-9]{2})', 1) END),
         x -> x IS NOT NULL AND x <> '')) AS ror_ids,
       -- the other creators are told apart by position (two creators may share a name); null and empty entries dropped, so
       -- the NOT ARRAY_CONTAINS / NOT ARRAYS_OVERLAP guards below never meet a NULL (which would drop the row)
       ARRAY_DISTINCT(FILTER(FLATTEN(TRANSFORM(FILTER(creators, (x, i) -> i <> pos),
         x -> TRANSFORM(COALESCE(x.affiliation, ARRAY()),
                        a -> REGEXP_EXTRACT(LOWER(COALESCE(a.affiliationIdentifier, '')), '(0[a-z0-9]{6}[0-9]{2})', 1)))),
         y -> y IS NOT NULL AND y <> '')) AS other_affiliation_rors,
       ARRAY_DISTINCT(FILTER(FLATTEN(TRANSFORM(FILTER(creators, (x, i) -> i <> pos),
         x -> TRANSFORM(COALESCE(x.affiliation, ARRAY()), a -> TRIM(REGEXP_REPLACE(LOWER(a.name), '[^\\p{L}\\p{N}]+', ' '))))),
         y -> y IS NOT NULL AND y <> '')) AS other_affiliation_names
FROM latest LATERAL VIEW POSEXPLODE(creators) t AS pos, c
WHERE rn = 1 AND c.nameType = 'Organizational'
  AND EXISTS(c.nameIdentifiers, i -> LOWER(i.nameIdentifierScheme) = 'ror')
  AND NOT EXISTS(c.nameIdentifiers, i -> LOWER(i.nameIdentifierScheme) = 'orcid')

In [ ]:
-- oxjob #2437: the served seat each ROR creator is: the work with that DOI in tonight's base, and the seat whose printed
-- name is the creator's name once lowercased, with punctuation and spacing folded. Kept only when that name is one of the
-- ROR record's own names (the ROR id names the creator, not an affiliation: 99.0% of seats, 21 Nov 2026), no other creator
-- of the record names the organization as their affiliation, and the record comes from a DataCite client whose ROR
-- creators passed the bar: DiSSCo's specimen records (ylqb.ybhfwy; 800 of 800 right, every institution checked). Other
-- clients failed (83%: institutional repositories list the university as the creator of theses and articles its people
-- wrote), so their seats stay untyped.
CREATE OR REPLACE TABLE openalex.authors.org_byline_ror_seats
COMMENT 'oxjob #2437: byline seats matched to a DataCite ROR creator whose ROR record names it. Rebuilt by TypeOrganizationSeats each run.'
AS
WITH ror_names AS (
  SELECT REGEXP_EXTRACT(LOWER(r.id), '(0[a-z0-9]{6}[0-9]{2})', 1) AS ror_id, TRIM(REGEXP_REPLACE(LOWER(n.value), '[^\\p{L}\\p{N}]+', ' ')) AS name_norm
  FROM openalex.institutions.ror r LATERAL VIEW EXPLODE(r.names) t AS n
),
cr AS (
  SELECT d.doi, d.creator_position, d.ror_ids, TRIM(REGEXP_REPLACE(LOWER(d.creator_name), '[^\\p{L}\\p{N}]+', ' ')) AS creator_norm
  FROM openalex.authors.org_byline_ror_creators d
  WHERE d.client_id IN ('ylqb.ybhfwy')
    AND NOT ARRAYS_OVERLAP(d.ror_ids, d.other_affiliation_rors)
    AND NOT ARRAY_CONTAINS(d.other_affiliation_names, TRIM(REGEXP_REPLACE(LOWER(d.creator_name), '[^\\p{L}\\p{N}]+', ' ')))
),
named_by_ror AS (
  SELECT DISTINCT x.doi, x.creator_position, x.rid AS ror_id
  FROM (SELECT c.doi, c.creator_position, c.creator_norm, rid FROM cr c LATERAL VIEW EXPLODE(c.ror_ids) e AS rid) x
  JOIN ror_names rn ON rn.ror_id = x.rid AND rn.name_norm = x.creator_norm
)
SELECT b.id AS work_id, CAST(p.pos AS BIGINT) AS author_sequence, a.raw_author_name, cr.doi, cr.creator_position,
       MIN(nr.ror_id) AS ror_id
FROM cr
JOIN named_by_ror nr ON nr.doi = cr.doi AND nr.creator_position = cr.creator_position
JOIN openalex.works.openalex_works_base b ON LOWER(b.doi) = CONCAT('https://doi.org/', cr.doi)
LATERAL VIEW POSEXPLODE(b.authorships) p AS pos, a
WHERE TRIM(REGEXP_REPLACE(LOWER(a.raw_author_name), '[^\\p{L}\\p{N}]+', ' ')) = cr.creator_norm
GROUP BY b.id, p.pos, a.raw_author_name, cr.doi, cr.creator_position

In [ ]:
-- oxjob #2437: the guards' verdict per distinct seat name (ror = TRUE: placeholders, person signs, a person beside the group,
-- fragments, a person then an affiliation, and initials-led names are refused). The only statement that calls the UDF.
CREATE OR REPLACE TABLE openalex.authors.org_byline_ror_verdicts
COMMENT 'oxjob #2437: org-names-v2 verdict per distinct ROR-matched seat name. Rebuilt by TypeOrganizationSeats each run.'
AS
SELECT raw_author_name, openalex.authors.type_org_byline_name(raw_author_name, TRUE, TRUE) AS v
FROM (SELECT DISTINCT raw_author_name FROM openalex.authors.org_byline_ror_seats WHERE raw_author_name IS NOT NULL)

In [ ]:
-- oxjob #2437: group-name seats: every seat of tonight's base whose printed name (exactly) has the confirmer's answer
-- organization, the name's latest answer deciding (a later answer of another label withdraws it), and that carries no
-- ORCID of its own (an ORCID marks a person). ~88K names, broadcast; past ~1M names, drop the hint.
CREATE OR REPLACE TABLE openalex.authors.org_byline_name_seats
COMMENT 'oxjob #2437: byline seats whose printed name the organization name confirmer answered organization. Rebuilt by TypeOrganizationSeats each run.'
AS
SELECT /*+ BROADCAST(c) */ s.work_id, s.author_sequence, s.raw_author_name, c.model, c.rubric
FROM (SELECT b.id AS work_id, CAST(p.pos AS BIGINT) AS author_sequence, a.raw_author_name
      FROM openalex.works.openalex_works_base b LATERAL VIEW POSEXPLODE(b.authorships) p AS pos, a
      WHERE a.raw_orcid IS NULL) s
JOIN (SELECT raw_author_name, label, model, rubric FROM openalex.authors.org_name_confirmations
      QUALIFY ROW_NUMBER() OVER (PARTITION BY raw_author_name ORDER BY confirmed_at DESC, label) = 1) c
  ON c.raw_author_name = s.raw_author_name
WHERE c.label = 'organization'

In [ ]:
-- oxjob #2437: this run's organization seats, one row per seat (ROR first, then confirmed group names), minus seats the nightly
-- confirmer has typed software-agent.
CREATE OR REPLACE TABLE openalex.works.work_author_types_org_staging
COMMENT 'oxjob #2437: organization seats (DataCite ROR creators, confirmed group names), before the guarded write. Rebuilt by TypeOrganizationSeats each run.'
AS
WITH ror AS (
  SELECT s.work_id, s.author_sequence, MIN(s.raw_author_name) AS raw_author_name, 'ror' AS rule,
         'org-names-v2+ror-dissco' AS rule_version,
         TO_JSON(NAMED_STRUCT('source', 'datacite', 'doi', MIN(s.doi), 'name_identifier_scheme', 'ROR', 'ror', MIN(s.ror_id))) AS evidence
  FROM openalex.authors.org_byline_ror_seats s
  JOIN openalex.authors.org_byline_ror_verdicts n ON n.raw_author_name = s.raw_author_name
  WHERE n.v.type = 'organization' AND n.v.rule = 'ror'
  GROUP BY s.work_id, s.author_sequence
),
named AS (
  SELECT s.work_id, s.author_sequence, MIN(s.raw_author_name) AS raw_author_name, 'group_name_confirmed' AS rule,
         'org-names-v2+confirm-gemini-4-hydrogen' AS rule_version,
         TO_JSON(NAMED_STRUCT('source', 'byline_name', 'model', MIN(s.model), 'rubric', MIN(s.rubric))) AS evidence
  FROM openalex.authors.org_byline_name_seats s
  LEFT ANTI JOIN ror r ON r.work_id = s.work_id AND r.author_sequence = s.author_sequence
  GROUP BY s.work_id, s.author_sequence
)
SELECT x.work_id, x.author_sequence, x.raw_author_name, 'organization' AS type, x.rule, x.rule_version, x.evidence
FROM (SELECT * FROM ror UNION ALL SELECT * FROM named) x
LEFT ANTI JOIN (SELECT work_id, author_sequence FROM openalex.works.work_author_types WHERE type = 'software-agent') a
  ON a.work_id = x.work_id AND a.author_sequence = x.author_sequence

In [ ]:
-- oxjob #2437: how far this run moves the organization rows, for each rule on its own (a big ROR class must not hide a
-- name-class accident). Over a rule's lines (lost max(10,000, 10%), added max(100,000, 10%) of that rule's rows now) the
-- run stops here and the rows stay; a planned change (the first fill) runs with allow_large_change = true.
DECLARE OR REPLACE VARIABLE ror_now BIGINT;
DECLARE OR REPLACE VARIABLE ror_lost BIGINT;
DECLARE OR REPLACE VARIABLE ror_added BIGINT;
DECLARE OR REPLACE VARIABLE nam_now BIGINT;
DECLARE OR REPLACE VARIABLE nam_lost BIGINT;
DECLARE OR REPLACE VARIABLE nam_added BIGINT;
SET VARIABLE ror_now = (SELECT COUNT(*) FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'ror');
SET VARIABLE nam_now = (SELECT COUNT(*) FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'group_name_confirmed');
SET VARIABLE ror_lost = (SELECT COUNT(*) FROM (SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'ror'
                                              EXCEPT ALL
                                              SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types_org_staging WHERE rule = 'ror'));
SET VARIABLE ror_added = (SELECT COUNT(*) FROM (SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types_org_staging WHERE rule = 'ror'
                                               EXCEPT ALL
                                               SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'ror'));
SET VARIABLE nam_lost = (SELECT COUNT(*) FROM (SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'group_name_confirmed'
                                              EXCEPT ALL
                                              SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types_org_staging WHERE rule = 'group_name_confirmed'));
SET VARIABLE nam_added = (SELECT COUNT(*) FROM (SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types_org_staging WHERE rule = 'group_name_confirmed'
                                               EXCEPT ALL
                                               SELECT work_id, author_sequence, raw_author_name FROM openalex.works.work_author_types WHERE type = 'organization' AND rule = 'group_name_confirmed'));
SELECT ror_now, ror_lost, ror_added, nam_now, nam_lost, nam_added;
SELECT CASE
  WHEN NOT openalex.works.e2e_gate('guardrails_override', 'work_author_types_org', ror_lost, GREATEST(10000, ror_now DIV 10), :allow_large_change)
    OR NOT openalex.works.e2e_gate('guardrails_override', 'work_author_types_org_added', ror_added, GREATEST(100000, ror_now DIV 10), :allow_large_change)
    OR NOT openalex.works.e2e_gate('guardrails_override', 'work_author_types_org_names', nam_lost, GREATEST(10000, nam_now DIV 10), :allow_large_change)
    OR NOT openalex.works.e2e_gate('guardrails_override', 'work_author_types_org_names_added', nam_added, GREATEST(100000, nam_now DIV 10), :allow_large_change)
  THEN RAISE_ERROR(CONCAT('oxjob #2437: organization rows over the lines, nothing written: ror lose ', ror_lost, ' add ', ror_added,
                          ' (now ', ror_now, '); names lose ', nam_lost, ' add ', nam_added, ' (now ', nam_now,
                          '). If planned, run with allow_large_change = true.'))
END

In [ ]:
-- oxjob #2437: replace the organization rows only. The software-agent check is repeated inside this statement, so a seat the
-- nightly confirmer typed after staging was built is still left alone (Authorships also leaves a seat untyped when two
-- rows disagree).
INSERT INTO openalex.works.work_author_types REPLACE WHERE type = 'organization'
SELECT o.work_id, o.author_sequence, o.raw_author_name, o.type, o.rule, o.rule_version, o.evidence
FROM openalex.works.work_author_types_org_staging o
LEFT ANTI JOIN (SELECT work_id, author_sequence FROM openalex.works.work_author_types WHERE type = 'software-agent') a
  ON a.work_id = o.work_id AND a.author_sequence = o.author_sequence

In [ ]:
-- oxjob #2437: what the table holds now (display only)
SELECT type, rule, rule_version, COUNT(*) AS seats, COUNT(DISTINCT work_id) AS works
FROM openalex.works.work_author_types GROUP BY type, rule, rule_version ORDER BY seats DESC